# 6. Distribución de productos

Una empresa fabrica televisores en las ciudades de Portland y Baltimore, y debe abastecer un cierto número de unidades a sus centros de distribución en Las Vegas y Detroit. Los costos de envı́o se resumen en la Tabla 4 y los datos de oferta y demanda, expresados en número de unidades, en la Tabla 5.

**a)** Construya un modelo de programación lineal entera que determine cómo deben realizarse los envı́os desde Portland y Baltimore para minimizar los costos de distribución.

In [42]:
from docplex.mp.model import Model
import numpy as np

mdl = Model(name='Distribución de productos')

## Datos y supuestos  
* Produccion: Portland, Balitimore
* Distribucion: Las Vegas, Detroit
* Costos de envío por unidad en Tabla 4
* Unidades disponibles y Demanda en Tabla 5
* cómo deben realizarse los envı́os desde Portland y Baltimore?
* minimizar los costos de distribución

## Formulación matemática 
### Variables de decisión
* cómo deben realizarse los envı́os desde Portland y Baltimore?
Unidades de la planta $i$ asignadas al centro de distribucion $j$
$$x_{ij}$$
$$i \epsilon \{Portland, Baltimore\} = \{P, B\}$$
$$j \epsilon \{Las Vegas, Detroit\} = \{V, D\}$$


In [43]:
productoras = ['P', 'B']
distribuidoras = ['V', 'D']

productoras_i = range(len(productoras))   # [0, 1, 2, 3, 4]
distribuidoras_j = range(len(distribuidoras)) # [0, 1, 2, 3, 4]

x = mdl.integer_var_matrix(productoras, distribuidoras, name='x')

In [44]:
x

{('P', 'V'): docplex.mp.Var(type=I,name='x_P_V'),
 ('P', 'D'): docplex.mp.Var(type=I,name='x_P_D'),
 ('B', 'V'): docplex.mp.Var(type=I,name='x_B_V'),
 ('B', 'D'): docplex.mp.Var(type=I,name='x_B_D')}

In [45]:
x['P', 'V']

docplex.mp.Var(type=I,name='x_P_V')

### Restricciones

In [46]:
# datos asociados a las productoras y distribuidoras
oferta = {
    'P': 200,   # Portland
    'B': 600    # Baltimore
}

demanda = {
    'V': 300,   # Las Vegas
    'D': 400    # Detroit
}

#### Restricciones 1: Oferta 
* Unidades disponibles y Demanda en Tabla 5

Portland
$$ x_{PV} + x_{PD} \leq 200 $$

Baltimore
$$ x_{BV} + x_{BD} \leq 600 $$

In [47]:
# oferta
mdl.add_constraints(
    mdl.sum(x[i, j] for j in distribuidoras) <= oferta[i]
    for i in productoras
)

[docplex.mp.LinearConstraint[](x_P_V+x_P_D,LE,200),
 docplex.mp.LinearConstraint[](x_B_V+x_B_D,LE,600)]

#### Restricciones 2: Demanda
* Unidades disponibles y Demanda en Tabla 5

Las Vegas
$$ x_{PV} + x_{BV} \ge 300 $$

Detroit
$$ x_{PD} + x_{BD} \ge 400 $$

In [48]:
# demanda
mdl.add_constraints(
    mdl.sum(x[i, j] for i in productoras) >= demanda[j]
    for j in distribuidoras
)

[docplex.mp.LinearConstraint[](x_P_V+x_B_V,GE,300),
 docplex.mp.LinearConstraint[](x_P_D+x_B_D,GE,400)]

## Función objetivo 
* minimizar los costos de distribución
$$\min z= 30x_{PV} + 40x_{PD} + 60x_{BV} + 50x_{BD}​$$

In [ ]:
# costos de transporte c[i, j]
costos = {
    ('P', 'V'): 30,
    ('P', 'D'): 40,
    ('B', 'V'): 60, 
    ('B', 'D'): 50  
}

# Función Objetivo 
mdl.minimize(
    mdl.sum(costos[i, j] * x[i, j] for i in productoras for j in distribuidoras)
)

In [50]:
# Resolver e imprimir solucion
solution = mdl.solve(log_output=True,)


print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 32000.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 2 times.
MIP Presolve eliminated 2 rows and 2 columns.
Aggregator did 2 substitutions.
All rows and columns eliminated.
Presolve time = 0.02 sec. (0.01 ticks)

Root node processing (before b&c):
  Real time             =    0.02 sec. (0.01 ticks)
Parallel b&c, 4 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.02 sec. (0.01 ticks)
solution for: Distribución de productos
objective: 32000
status: OPTIMAL_SOLUTION(2)
x_P_V=200
x_B_V=100
x_B_D=400

